# Phase 6: Explainable AI (XAI) using SHAP & LIME

**Objective:** Unbox the champion XGBoost model and transform it into an interpretable decision support system.

### Core XAI Capabilities in this Phase:
1. **Global Explainability (SHAP Beeswarm & Bar Plots):** Discovers which features consistently push loan risk up or down across the entire portfolio.
2. **Local Explainability (SHAP Waterfall Plots):** Breaks down the exact risk drivers for individual loan applicants (high-risk vs. low-risk).
3. **LIME (Local Interpretable Model-agnostic Explanations):** Validates individual predictions using a local surrogate model.

### Step 6.1: Import Libraries and Load Saved Model Artifacts

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import joblib
import warnings

# Explainable AI Libraries
import shap
from lime import lime_tabular
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

# Load trained model artifacts
MODELS_DIR = Path("../models") if Path("../models").exists() else Path("models")
model = joblib.load(MODELS_DIR / "best_model.pkl")
feature_names = joblib.load(MODELS_DIR / "feature_names.pkl")

# Load feature dataset
FEAT_PATH = Path("../data/train_features.csv") if Path("../data/train_features.csv").exists() else Path("data/train_features.csv")
df = pd.read_csv(FEAT_PATH)

X = df[feature_names]
y = df['loan_default']

# Stratified 80/20 split matching Phase 4
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Sample 1,000 validation cases for fast and representative SHAP computation
X_sample = X_val.sample(n=1000, random_state=42)
y_sample = y_val.loc[X_sample.index]

print(f"Model loaded: {type(model).__name__}")
print(f"SHAP evaluation sample: {X_sample.shape[0]:,} applicants x {X_sample.shape[1]} features")

### Step 6.2: Compute SHAP Values (TreeExplainer)

> **Viva Insight:** SHAP (SHapley Additive exPlanations) is based on cooperative game theory. It guarantees that the sum of feature contributions equals the difference between the model's actual prediction and the average prediction (efficiency property).

In [ ]:
# Initialize TreeExplainer for XGBoost
explainer = shap.TreeExplainer(model)

# Compute SHAP values on the sample
shap_values = explainer(X_sample)

print(f"SHAP values shape: {shap_values.shape}")
print(f"Base portfolio risk (log-odds E[f(x)]): {explainer.expected_value:.4f}")

### Step 6.3: Global Interpretability - SHAP Beeswarm Summary Plot

**How to interpret:**
- **Y-axis:** Features ordered from most impactful to least impactful.
- **X-axis:** Impact on model output (log-odds). Positive $\rightarrow$ increases default risk; Negative $\rightarrow$ decreases default risk.
- **Color:** Value of the feature for that applicant (**Red = High**, **Blue = Low**).

In [ ]:
plt.figure(figsize=(10, 6))
shap.plots.beeswarm(shap_values, max_display=12, show=False)
plt.title("SHAP Global Summary: Feature Impact on Loan Default Risk", fontsize=13, pad=12)
plt.tight_layout()
plt.show()

### Step 6.4: Global Mean Importance Bar Plot

Ranks features by their mean absolute SHAP value: $\text{mean}(|\text{SHAP value}|)$ across all applicants.

In [ ]:
plt.figure(figsize=(10, 5))
shap.plots.bar(shap_values, max_display=12, show=False)
plt.title("Top 12 Global Features Ranked by Average SHAP Impact", fontsize=13, pad=12)
plt.tight_layout()
plt.show()

### Step 6.5: Local Interpretability - Explaining an Individual High-Risk Borrower (Waterfall Plot)

We select an applicant flagged as **High Risk** ($P \ge 0.50$) to show the adverse action reasons explaining why the loan was flagged.

In [ ]:
# Find an applicant with high predicted default probability
sample_probas = model.predict_proba(X_sample)[:, 1]
high_risk_idx = np.where(sample_probas >= 0.65)[0][0]

print(f"High-Risk Applicant Index: {high_risk_idx}")
print(f"Predicted Default Probability: {sample_probas[high_risk_idx]:.2%}")
print(f"Actual Status: {'Defaulted (1)' if y_sample.iloc[high_risk_idx] == 1 else 'Non-Default (0)'}")

# Generate SHAP Waterfall Plot
plt.figure(figsize=(10, 6))
shap.plots.waterfall(shap_values[high_risk_idx], max_display=10, show=False)
plt.title(f"Adverse Action Explanation: High-Risk Applicant (P={sample_probas[high_risk_idx]:.1%})", fontsize=12, pad=12)
plt.tight_layout()
plt.show()

### Step 6.6: Local Interpretability - Explaining an Individual Low-Risk Borrower (Waterfall Plot)

We select a creditworthy applicant approved with very low risk ($P < 0.25$) to observe protective factors.

In [ ]:
# Find an applicant with low predicted default probability
low_risk_idx = np.where(sample_probas <= 0.25)[0][0]

print(f"Low-Risk Applicant Index: {low_risk_idx}")
print(f"Predicted Default Probability: {sample_probas[low_risk_idx]:.2%}")
print(f"Actual Status: {'Non-Default (0)' if y_sample.iloc[low_risk_idx] == 0 else 'Defaulted (1)'}")

# Generate SHAP Waterfall Plot
plt.figure(figsize=(10, 6))
shap.plots.waterfall(shap_values[low_risk_idx], max_display=10, show=False)
plt.title(f"Approval Explanation: Low-Risk Applicant (P={sample_probas[low_risk_idx]:.1%})", fontsize=12, pad=12)
plt.tight_layout()
plt.show()

### Step 6.7: Local Surrogate Explanations with LIME

> **Viva Insight:** LIME (Local Interpretable Model-agnostic Explanations) perturbs the applicant's attributes in a local neighborhood and fits an interpretable sparse linear model to approximate the complex model's boundary locally.

In [ ]:
# Initialize LIME Tabular Explainer
lime_explainer = lime_tabular.LimeTabularExplainer(
    training_data=np.array(X_train.sample(1000, random_state=42)),
    feature_names=list(feature_names),
    class_names=['Non-Default', 'Default'],
    mode='classification',
    random_state=42
)

# Explain the same high-risk applicant using LIME
lime_exp = lime_explainer.explain_instance(
    data_row=X_sample.iloc[high_risk_idx].values,
    predict_fn=model.predict_proba,
    num_features=8
)

print("--- LIME Local Decision Rules ---")
for rule, weight in lime_exp.as_list():
    direction = "Increases Default Risk (Red)" if weight > 0 else "Decreases Default Risk (Green)"
    print(f"{rule:45s} | Weight: {weight:+.4f} | {direction}")

# Plot LIME explanation
fig = lime_exp.as_pyplot_figure()
plt.title("LIME Explanation for High-Risk Applicant", fontsize=12, pad=10)
plt.tight_layout()
plt.show()